# 19 - בניית גרפים לפי שעות היום

כל רשת שנבנתה עד כה בפרויקט זה מכווצת את כל לוח הזמנים ל**גרף סטטי אחד**: קטע `u -> v` קיים אם *איזושהי* נסיעה, ב*כל* שעה, ב*כל* יום, עוברת מ-`u` ל-`v`. זהו מודל ראשוני שימושי, אך הוא מסתיר את העובדה שרשת תחבורה ציבורית איננה רשת אחת - היא רשת שונה מדי כמה שעות. תחנה המהווה נקודת חיתוך קריטית בשעה 03:00 עשויה להיות אחת מבין חלופות רבות ועודפות בשעה 08:00.

מחברת זו היא **חלק 1 של כיוון המחקר העתידי 1**. היא פורסת את לוח הזמנים לחמישה חלונות זמן ובונה גרף נפרד לכל אחד מהם, במעבר streaming *יחיד* על הקובץ `stop_times.txt` בגודל 816 MB. בנוסף היא מכניסה לשימוש את `calendar.txt` - קובץ GTFS שהפרויקט מעולם לא נגע בו - כך שניתן לתייג כל נסיעה כפועלת ב**ימי חול (ראשון-חמישי)**, ב**יום שישי**, ב**שבת**, או בשילוב כלשהו ביניהם. הבחנה זו מהותית ביותר בישראל, שבה מרבית שירותי האוטובוס והרכבת נעצרים בשבת, ולכן גרף יחיד של כלל השירות ממצע בשקט רשת מלאה של יום חול יחד עם רשת כמעט ריקה של שבת.

מחברת 20 צורכת את הגרפים הנכתבים כאן כדי להשוות centrality וחוסן בין החלונות.

**שאלת המחקר הנדונה כאן:** עד כמה באמת משתנה הטופולוגיה של רשת התחבורה הציבורית הישראלית לאורך היום ולאורך השבוע - האם הרשת מתכווצת, מתפרקת, או רק מידללת, ובאיזו מידה?

## הסתייגות החלה על כל מספר להלן

GTFS הוא **לוח זמנים**. כל מה שנספר במחברת זו הוא נסיעות *מתוכננות* ומעברי קטעים *מתוכננים*. אלה **אינם** נתוני נסועה (ridership), אינם תפוסת כלי רכב ואינם שירות שהתממש בפועל. קטע המשורת על ידי 40 אוטובוסים מתוכננים בשעת שיא הבוקר עשוי להסיע 4,000 נוסעים או 40; דבר בפיד זה אינו יכול לומר לנו מהו המצב. בכל מקום שבו הטקסט אומר *נפח שירות* הכוונה היא ל*יציאות מתוכננות*.

## קלט

* `israel-public-transportation/stop_times.txt` - 816 MB, 15.7M שורות, **אינו מנוהל ב-git**; מורד לפי דרישה מ-Google Drive על ידי התא שלהלן (אותו file id כמו במחברת 02).
* `israel-public-transportation/trips.txt` - ממפה `trip_id` ל-`service_id`.
* `israel-public-transportation/calendar.txt` - ממפה `service_id` לתבנית ימי השבוע שבה הוא פועל.
* `outputs/nb/02_graph_construction/tables/nodes.csv` - מאפייני תחנות (`stop_id, stop_name, lat, lon, region, metro`) המשמשים לעיטור הגרפים של כל חלון.

## פלט

הכל נכתב תחת `outputs/nb/19_time_of_day_graphs/`:

* `tables/window_summary.csv` - `window, start_hour, end_hour, trips, nodes, directed_edges, avg_degree, largest_component_share` (החוזה שמחברות אחרות תלויות בו).
* `tables/edges_<window>.csv` - קובץ אחד לכל חלון, `from_stop, to_stop, trip_frequency`.
* `graph_<window>.pkl` - גרף `networkx` **בלתי מכוון** אחד, בפיקל, לכל חלון.
* `tables/window_summary_by_daytype.csv` - אותן סטטיסטיקות בפילוח לשירות של יום חול / יום שישי / שבת.
* `tables/hourly_service_volume.csv` - יציאות מתוכננות מתחנות והתחלות נסיעה לכל שעה ביום השירות.
* `tables/service_calendar_summary.csv` - כמה שירותים וכמה נסיעות פועלים בכל יום בשבוע.
* `tables/window_edge_overlap.csv` - חפיפת Jaccard של קבוצות הקשתות עבור כל זוג חלונות.
* `figures/service_volume_by_hour.png`, `figures/network_size_by_window.png`, `figures/connectivity_by_window.png`, `figures/weekday_vs_weekend_by_window.png`, `figures/window_edge_overlap.png`.
* `window_construction_summary.json` - כל המספרים המרכזיים בתוספת סטטיסטיקות מעבר ה-streaming.

דבר מחוץ ל-`outputs/nb/19_time_of_day_graphs/` אינו נכתב. התיקיות המוקפאות המצוטטות בדוח - `outputs/tables`, `outputs/figures` ו-`outputs/rail` - אינן נוגעות כלל.

## מחברות שחייבות לרוץ קודם

* `02_graph_construction` - עבור `tables/nodes.csv`. (זו גם המחברת שמורידה לראשונה את `stop_times.txt`, אף שמחברת זו מסוגלת להוריד אותו בעצמה.)

אין צורך בשלב נוסף כלשהו.

## 1. אתחול סביבת העבודה

התא שלהלן מאפשר להריץ את המחברת הן על עותק מקומי והן על Google Colab. הוא מגדיר את `_ensure(...)`, המתקין באמצעות pip רק את החבילות החסרות בפועל (כך שהרצה חוזרת של המחברת זולה), ואת `find_repo_root()`, המטפסת מהתיקייה הנוכחית כלפי מעלה בחיפוש אחר תיקיית ה-GTFS, ובהיעדרה משכפלת את המאגר אל `/content`. לאחר מכן הוא מגדיר את `REPO`, `DATA` ו-`OUT` ויוצר את תיקיית הפלט של המחברת. כל תא מאוחר יותר מסתמך על שלושת הנתיבים הללו, ולכן תא זה חייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות שלב וקבועים הניתנים לכוונון

אנו מייבאים את ערכת הכלים המדעית ומקבעים את מבנה התיקיות: מחברת זו מחזיקה בדיוק תיקיית פלט אחת, `outputs/nb/19_time_of_day_graphs/`, ובה תת-התיקיות `tables/` ו-`figures/`.

כל ההגדרות היקרות או השנויות בבחירה מרוכזות כאן, כך שבודק יוכל לשנותן במקום אחד:

* **`WINDOWS`** היא הגדרת פרוסות הזמן, בצורת `(name, start_hour, end_hour)` כאשר `end_hour` אינו כלול ומותרת גלישה מסביב ליממה (`night` נמשך מ-23:00 עד 06:00). חמשת החלונות חייבים לרצף את כל 24 השעות בדיוק פעם אחת - התא הבא בודק זאת ב-assert, כך שניתן לערוך רשימה זו בחופשיות והמחברת תתריע מיד אם ההגדרה החדשה יוצרת חור או חפיפה.
* **`PROGRESS_EVERY`** קובע כל כמה זמן מדפיס מעבר ה-streaming התקדמות. **הערת עלות:** מעבר ה-streaming קורא את כל 15.7M השורות של הקובץ `stop_times.txt` בגודל 816 MB בדיוק **פעם אחת**, ואורך בדרך כלל **4-8 דקות**; זהו ללא ספק המרכיב הדומיננטי בעלות המחברת. הוא בונה את כל מוני החלונות בו-זמנית, כך שקריאה חוזרת של הקובץ לכל חלון (שהייתה עולה פי 5) לעולם אינה מתרחשת. שיא צריכת הזיכרון הוא כמה מאות MB - מיפוי הנסיעה-לשירות (כ-420k רשומות) בתוספת 15 מוני קשתות של כ-50k רשומות לכל היותר כל אחד.
* **`WRITE_WINDOW_PICKLES`** ניתן להגדרה כ-`False` כדי לדלג על כתיבת חמשת הקבצים `graph_<window>.pkl` (כמה MB כל אחד). מחברת 20 קוראת אותם, ולכן יש להשאיר `True` אלא אם דרושות רק הטבלאות.
* **`FIG_DPI`** ו-**`TOP_N`** משפיעים רק על גודל האיורים ועל אורך הטבלאות.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn')

import csv, json, pickle, time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

# A handful of rows in stop_times.txt are very long; raise the csv field limit up front.
csv.field_size_limit(10_000_000)

STAGE = OUT / '19_time_of_day_graphs'    # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
# (name, start_hour, end_hour) with end_hour EXCLUSIVE; a window may wrap past midnight.
WINDOWS = [
    ('morning_peak',   6,  9),
    ('midday',         9, 15),
    ('afternoon_peak', 15, 19),
    ('evening',        19, 23),
    ('night',          23,  6),
]

PROGRESS_EVERY = 2_000_000     # rows between progress prints in the single streaming pass
WRITE_WINDOW_PICKLES = True    # write graph_<window>.pkl (notebook 20 needs these)
FIG_DPI = 150                  # figure resolution; drop to 90 for faster, smaller files
TOP_N = 15                     # rows shown in preview tables

print('pandas', pd.__version__, '| networkx', nx.__version__)
print('this stage :', STAGE)

## 3. עיבוד תוויות בעברית

שמות התחנות בפיד ה-GTFS הישראלי הם בעברית, וטבלאות התצוגה המקדימה ואיור אחד להלן מדפיסים אותם. Matplotlib אינו ממש את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן טקסט מימין לשמאל יוצא הפוך ובלתי קריא. התא שלהלן מבצע monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה תווים עבריים מומרת לסדר תצוגה באמצעות `python-bidi` לפני שהיא מצוירת, ובוחר גופן שאכן מכיל גליפים עבריים (Arial ב-Windows, DejaVu Sans בכל מקום אחר). התא הוא אידמפוטנטי - הרצה חוזרת שלו לא תערים patch על patch. כל שאר הטקסט במחברת הוא באנגלית, בהתאם לדרישת ההגשה.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור תוצרים של שלבים קודמים

מחברת זו זקוקה לתוצר קודם אחד בלבד: `nodes.csv` ממחברת 02, המספק את שמות התחנות, הקואורדינטות והאזורים שאנו מצרפים לכל גרף חלון. תיקיות השלבים מזוהות לפי **הקידומת הדו-ספרתית** שלהן ולא לפי ה-slug המלא, כך שתיקייה ששמה שונה (`02_graph_construction` לעומת `02_graphs`) עדיין תזוהה. אם התיקייה או הקובץ חסרים אנו זורקים `FileNotFoundError` המציין במפורש את המחברת שיש להריץ תחילה, במקום כישלון מאוחר יותר עם `KeyError` סתום.

In [ ]:
# --- Stage resolution helpers ---------------------------------------------
def stage_dir(prefix, notebook_hint):
    """Return the output folder whose name starts with `prefix` (e.g. '02')."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            'No stage folder starting with ' + repr(prefix) + ' under ' + str(OUT) +
            ' - run notebook ' + notebook_hint + ' first.'
        )
    return matches[0]


def load_stage_table(prefix, filename, notebook_hint, **read_kwargs):
    """Load a CSV from an earlier stage, searching the stage folder recursively."""
    folder = stage_dir(prefix, notebook_hint)
    direct = folder / filename
    if direct.exists():
        path = direct
    else:
        found = sorted(folder.rglob(filename))
        if not found:
            raise FileNotFoundError(
                filename + ' not found anywhere under ' + str(folder) +
                ' - run notebook ' + notebook_hint + ' first; it writes ' + filename + '.'
            )
        path = found[0]
    df = pd.read_csv(path, encoding='utf-8-sig', **read_kwargs)
    print('loaded ' + filename + ': ' + format(len(df), ',') + ' rows  <-  ' + str(path))
    return df


def require_raw(filename):
    """Return the path of a raw GTFS file, or raise a clear error."""
    path = DATA / filename
    if not path.exists():
        raise FileNotFoundError(
            str(path) + ' is missing - the raw GTFS feed must be present in ' + str(DATA) + '.'
        )
    return path


nodes_df = load_stage_table('02', 'nodes.csv', '02_graph_construction',
                            dtype={'stop_id': str})
nodes_df.head(3)

## 5. הורדת `stop_times.txt`

הקובץ `stop_times.txt` הוא בגודל 816 MB ובמכוון אינו מנוהל ב-git. התא שלהלן מוריד אותו מ-Google Drive לפי דרישה, תוך שימוש בדיוק באותו file id כמו במחברת 02, ואינו עושה דבר אם הקובץ כבר קיים. שום דבר אחר בפיד אינו דורש הורדה - `trips.txt` ו-`calendar.txt` קטנים ונמצאים במאגר.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 6. חלונות זמן ופענוח זמני GTFS

שני דברים מוגדרים כאן.

**טבלת ההמרה משעה לחלון.** `WINDOWS` מורחב לטבלה בת 24 רשומות הממפה כל שעה ביום השירות לחלון אחד בדיוק. ההרחבה מוודאת ב-assert שהחלונות *מרצפים* את היממה - אף שעה אינה שייכת לשני חלונות, אף שעה אינה נותרת בחוץ - כך ששינוי בקבוע `WINDOWS` ייכשל ברעש ולא ישמיט קטעים בשקט.

**פענוח זמני GTFS.** זמני GTFS נמדדים מ*חצות השירות*, ולא מחצות שעון הקיר, ומותר להם לחרוג מ-24 שעות: `25:30:00` פירושו 01:30 בבוקר הקלנדרי שלמחרת, אך עדיין שייך ליום השירות *הקודם*. כ-1.1% מהשורות בפיד זה הן מסוג זה. העברת מחרוזת כזו ל-`datetime.strptime` זורקת `ValueError: unconverted data remains` ומקריסה את המעבר, ולכן איננו משתמשים ב-`datetime` כלל - `gtfs_seconds` מפצלת לפי `:` ומחזירה `h*3600 + m*60 + s`. כדי לשבץ יציאה בשעה 25:30 בחלון `night` אנו לוקחים את השעה **מודולו 24**, וזו הסמנטיקה המכוונת: אוטובוס של 25:30 הוא אוטובוס של 01:30 מבחינת איך שהרשת נראית באותו רגע. ה-assertions שלהלן מקבעים התנהגות זו.

In [ ]:
# --- Expand WINDOWS into an hour -> window lookup --------------------------
def window_hours(start, end):
    """Hours covered by [start, end); wraps past midnight when end <= start."""
    span = (end - start) % 24 or 24
    return [(start + i) % 24 for i in range(span)]


WINDOW_NAMES = [name for name, _, _ in WINDOWS]
WINDOW_INDEX = {name: i for i, name in enumerate(WINDOW_NAMES)}

_hour_owner = {}
for _name, _s, _e in WINDOWS:
    for _h in window_hours(_s, _e):
        if _h in _hour_owner:
            raise ValueError(
                'WINDOWS overlap: hour ' + str(_h) + ' is claimed by both ' +
                _hour_owner[_h] + ' and ' + _name + '.'
            )
        _hour_owner[_h] = _name
_gaps = [h for h in range(24) if h not in _hour_owner]
if _gaps:
    raise ValueError('WINDOWS leave hours uncovered: ' + str(_gaps))

# Plain python list -> fastest possible lookup inside the 15.7M-row loop.
HOUR_TO_WIDX = [WINDOW_INDEX[_hour_owner[h]] for h in range(24)]
HOUR_TO_WINDOW = {h: _hour_owner[h] for h in range(24)}


def gtfs_seconds(value):
    """Parse an HH:MM:SS GTFS time into seconds since SERVICE midnight.

    Hours may be >= 24 (25:30:00 = 01:30 on the next calendar day, same service day).
    Never use datetime/strptime here - it raises on hours >= 24.
    Returns None for blank or malformed values.
    """
    if not value:
        return None
    parts = value.split(':')
    if len(parts) != 3:
        return None
    try:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    except ValueError:
        return None


# Sanity checks on the two tricky cases.
assert gtfs_seconds('05:10:00') == 5 * 3600 + 10 * 60
assert gtfs_seconds('25:30:00') == 25 * 3600 + 30 * 60
assert (gtfs_seconds('25:30:00') // 3600) % 24 == 1
assert gtfs_seconds('') is None and gtfs_seconds('nonsense') is None

print('hour -> window map:')
for name, s, e in WINDOWS:
    hrs = window_hours(s, e)
    print('  ' + name.ljust(16) + str(s).rjust(2) + ':00 -> ' + str(e).rjust(2) +
          ':00  (' + str(len(hrs)) + ' hours: ' + ', '.join(str(h) for h in hrs) + ')')

## 7. `calendar.txt` - באיזה יום בשבוע פועל כל שירות?

זו הפעם הראשונה שהפרויקט נוגע ב-`calendar.txt`. ב-GTFS, כל נסיעה נושאת `service_id`, ו-`calendar.txt` מעניק לכל שירות שבע עמודות בוליאניות (`sunday` ... `saturday`) בתוספת טווח תאריכי תוקף. שבוע העבודה הישראלי נמשך **מיום ראשון עד יום חמישי**; **יום שישי** הוא יום קצר שבו השירות דועך במהלך אחר הצהריים, וב**שבת** רובה המכריע של תנועת האוטובוסים והרכבות אינה פועלת כלל.

לפיכך אנו מצמצמים כל שירות לשלושה משתנים בוליאניים בלתי תלויים:

* `runs_weekday` - פועל לפחות באחד מהימים ראשון-חמישי;
* `runs_friday`;
* `runs_saturday`.

אלה **אינם** זרים הדדית, וזאת במכוון: שירות בעל התבנית `1111110` אכן פועל גם בימי חול וגם ביום שישי, ויהיה זה שגוי לכפות עליו קטגוריה אחת. כתוצאה מכך, נסיעה עשויה להיספר ביותר מעמודת סוג-יום אחת בהמשך, ועמודות סוג-היום **אינן** מסתכמות בסך הכול. התא מדפיס את מפקד תבניות הימים כך שהדבר גלוי לעין ולא מונח כמובן מאליו.

הסתייגות אחת בכנות: למספר קטן של שירותים כל שבעת דגלי הימים מוגדרים כ-`0`. בפיד GTFS מלא, שירותים אלה היו מונעים כולם על ידי `calendar_dates.txt` (תאריכי שירות שנוספו במפורש), אך **פיד זה אינו כולל `calendar_dates.txt`**, ולכן עבור אותם שירותים איננו יודעים כלל באיזה יום הם פועלים. הם מדווחים בנפרד כ-`unknown` ועדיין נכללים בגרפים של כלל השירות.

In [ ]:
# --- Read calendar.txt and reduce every service to day-type flags ----------
DAY_COLS = ['sunday', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday']
WEEKDAY_COLS = ['sunday', 'monday', 'tuesday', 'wednesday', 'thursday']

# Bit codes, so the streaming pass can carry the day-type in a single small int.
BIT_WEEKDAY, BIT_FRIDAY, BIT_SATURDAY = 1, 2, 4

calendar_path = require_raw('calendar.txt')
service_code = {}          # service_id -> bit code
day_service_counts = Counter()
pattern_counts = Counter()

with open(calendar_path, encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        flags = {d: str(row.get(d, '0')).strip() == '1' for d in DAY_COLS}
        code = 0
        if any(flags[d] for d in WEEKDAY_COLS):
            code |= BIT_WEEKDAY
        if flags['friday']:
            code |= BIT_FRIDAY
        if flags['saturday']:
            code |= BIT_SATURDAY
        service_code[str(row['service_id']).strip()] = code
        for d in DAY_COLS:
            if flags[d]:
                day_service_counts[d] += 1
        pattern_counts[''.join('1' if flags[d] else '0' for d in DAY_COLS)] += 1

n_services = len(service_code)
n_unknown_services = sum(1 for c in service_code.values() if c == 0)
print('services in calendar.txt      : ' + format(n_services, ','))
print('services with no day flag set : ' + format(n_unknown_services, ',') +
      '  (no calendar_dates.txt in this feed -> day of week unknown)')
print()
print('Most common weekly patterns (Sun Mon Tue Wed Thu Fri Sat):')
for pat, cnt in pattern_counts.most_common(10):
    print('  ' + ' '.join(pat) + '   ' + format(cnt, '>8,') + ' services')

## 8. `trips.txt` - הצמדת סוג-יום לכל נסיעה

`stop_times.txt` מכיר רק את `trip_id`, ולכן לפני המעבר הגדול אנו בונים מילון `trip_id -> קוד ביט של סוג-יום` על ידי חיבור `trips.txt` לקודי השירות מהתא הקודם. מילון זה (כ-420k רשומות) הוא האובייקט הגדול היחיד המוחזק בזיכרון מלבד מוני הקשתות.

התא מפיק גם את התצוגה של השבוע *במשקל נסיעות*, שהיא אינפורמטיבית הרבה יותר מזו שבמשקל שירותים: שירותים הם ישויות מנהליות בגדלים שונים באופן קיצוני, ואילו נסיעות הן מסעות רכב מתוכננים בפועל. הנתון של יום שבת המודפס כאן הוא הראיה המרכזית לכך שפילוח לפי סוג-יום הוא מהותי.

In [ ]:
# --- Map every trip_id to its day-type bit code ----------------------------
trips_path = require_raw('trips.txt')
trip_code = {}
trips_missing_service = 0

with open(trips_path, encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sid = str(row.get('service_id', '')).strip()
        if sid not in service_code:
            trips_missing_service += 1
        trip_code[str(row['trip_id']).strip()] = service_code.get(sid, 0)

n_trips_total = len(trip_code)
trip_day_counts = {
    'weekday_sun_thu': sum(1 for c in trip_code.values() if c & BIT_WEEKDAY),
    'friday': sum(1 for c in trip_code.values() if c & BIT_FRIDAY),
    'saturday': sum(1 for c in trip_code.values() if c & BIT_SATURDAY),
    'unknown': sum(1 for c in trip_code.values() if c == 0),
}

print('trips in trips.txt                 : ' + format(n_trips_total, ','))
print('trips whose service_id is missing  : ' + format(trips_missing_service, ','))
print()
print('Scheduled trips by day of week (a trip can count in more than one row):')
for label, cnt in trip_day_counts.items():
    share = 100.0 * cnt / n_trips_total if n_trips_total else 0.0
    print('  ' + label.ljust(18) + format(cnt, '>9,') + '   ' + format(share, '5.1f') + '% of all trips')

service_calendar = pd.DataFrame(
    [{'day': d, 'services_running': day_service_counts.get(d, 0)} for d in DAY_COLS]
)
service_calendar['is_israeli_weekend'] = service_calendar['day'].isin(['friday', 'saturday'])
service_calendar.to_csv(TABLES / 'service_calendar_summary.csv', index=False, encoding='utf-8-sig')
service_calendar

## 9. מעבר ה-streaming היחיד על `stop_times.txt`

זהו התא היקר, וזוהי החלטת התכנון המשמעותית ביותר: **מעבר אחד, כל המונים בבת אחת.**

הקובץ ממוין לפי `(trip_id, stop_sequence)` - מחברת 02 מאמתת זאת על פני הקובץ כולו - ולכן שתי שורות עוקבות הנושאות אותו `trip_id` הן תחנות עוקבות של אותה נסיעה, ומכאן קטע מכוון אחד. לכל זוג כזה אנו משייכים את הקטע לחלון על פי **זמן היציאה בתחנת המוצא** (השורה המוקדמת יותר), שהוא הרגע שבו כלי הרכב חוצה בפועל את הקטע. כל מונה שלהלן מתעדכן באותה איטרציה עצמה:

* `edges_all[w]` - ספירת מעברי קטעים לכל חלון, על פני כלל השירותים;
* `edges_wd[w]`, `edges_fr[w]`, `edges_sa[w]` - אותו דבר, בהגבלה לנסיעות שהשירות שלהן פועל בימי חול / יום שישי / שבת;
* `hour_*` - יציאות מתוכננות מתחנות לכל שעה ביום השירות, ולכל שעה וסוג-יום;
* `trip_start_*` - *התחלות* נסיעה לכל שעה (השורה הראשונה בכל בלוק נסיעה);
* `trip_window_mask` - מסכת 5 ביטים לכל נסיעה המתעדת אילו חלונות אותה נסיעה נגעה בהם, כך שנסיעה החוצה את גבול 09:00 נספרת בשני החלונות שהיא משרתת.

צריכת הזיכרון היא `O(|E| * windows + |T|)`, ולעולם לא `O(rows)`: אף שורה אינה נשמרת. `defaultdict(int)` הממופתח לפי `(from_stop, to_stop)` מחזיק לכל היותר כמה עשרות אלפי רשומות לכל חלון.

**עלות:** כ-15.7M שורות, בדרך כלל **4-8 דקות** על מחשב נייד. קריאת הקובץ פעם אחת לכל חלון הייתה עולה פי חמישה, וזה בדיוק מה שמבנה זה מונע.

שורות בעלות `departure_time` שאינו ניתן לפענוח או ריק נספרות ומדולגות, במקום לנחש את ערכן.

In [ ]:
def stream_window_edges(path, trip_code, progress_every=PROGRESS_EVERY):
    """One pass over stop_times.txt, building every per-window counter at once.

    A segment (prev_stop -> stop) is assigned to the window containing the
    DEPARTURE time at prev_stop. Times are seconds since service midnight and the
    hour is taken modulo 24, so a 25:30 departure lands in the window covering 01:00.
    """
    n_w = len(WINDOW_NAMES)
    edges_all = [defaultdict(int) for _ in range(n_w)]
    edges_wd = [defaultdict(int) for _ in range(n_w)]
    edges_fr = [defaultdict(int) for _ in range(n_w)]
    edges_sa = [defaultdict(int) for _ in range(n_w)]

    hour_all = [0] * 24
    hour_wd = [0] * 24
    hour_fr = [0] * 24
    hour_sa = [0] * 24
    start_all = [0] * 24
    start_wd = [0] * 24
    start_fr = [0] * 24
    start_sa = [0] * 24

    trip_window_mask = defaultdict(int)

    rows_read = 0
    segments = 0
    self_loops_skipped = 0
    bad_times = 0
    rows_past_midnight = 0
    trips_seen = 0
    trips_not_in_trips_txt = set()
    t0 = time.time()

    with open(path, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index('trip_id')
        si = header.index('stop_id')
        di = header.index('departure_time')

        prev_trip = None
        prev_stop = None
        prev_secs = None
        code = 0

        for row in reader:
            rows_read += 1
            trip = row[ti]
            stop = row[si]
            secs = gtfs_seconds(row[di])
            if secs is None:
                bad_times += 1
            elif secs >= 86400:
                rows_past_midnight += 1

            new_trip = trip != prev_trip
            if new_trip:
                trips_seen += 1
                if trip in trip_code:
                    code = trip_code[trip]
                else:
                    code = 0
                    if len(trips_not_in_trips_txt) < 50:
                        trips_not_in_trips_txt.add(trip)

            is_wd = bool(code & BIT_WEEKDAY)
            is_fr = bool(code & BIT_FRIDAY)
            is_sa = bool(code & BIT_SATURDAY)

            if secs is not None:
                h = (secs // 3600) % 24
                hour_all[h] += 1
                if is_wd:
                    hour_wd[h] += 1
                if is_fr:
                    hour_fr[h] += 1
                if is_sa:
                    hour_sa[h] += 1
                if new_trip:
                    start_all[h] += 1
                    if is_wd:
                        start_wd[h] += 1
                    if is_fr:
                        start_fr[h] += 1
                    if is_sa:
                        start_sa[h] += 1

            if not new_trip and prev_stop is not None and prev_secs is not None:
                if prev_stop != stop:
                    w = HOUR_TO_WIDX[(prev_secs // 3600) % 24]
                    key = (prev_stop, stop)
                    edges_all[w][key] += 1
                    if is_wd:
                        edges_wd[w][key] += 1
                    if is_fr:
                        edges_fr[w][key] += 1
                    if is_sa:
                        edges_sa[w][key] += 1
                    trip_window_mask[trip] |= (1 << w)
                    segments += 1
                else:
                    self_loops_skipped += 1

            prev_trip, prev_stop, prev_secs = trip, stop, secs

            if progress_every and rows_read % progress_every == 0:
                uniq = sum(len(d) for d in edges_all)
                print('    ' + format(rows_read, ',') + ' rows | ' + format(uniq, ',') +
                      ' window-segments | ' + format(time.time() - t0, ',.0f') + 's')

    stats = {
        'stop_times_rows': rows_read,
        'trip_blocks': trips_seen,
        'segments_assigned': segments,
        'self_loops_skipped': self_loops_skipped,
        'rows_with_unparseable_time': bad_times,
        'rows_with_hour_ge_24': rows_past_midnight,
        'trips_absent_from_trips_txt_sample': sorted(trips_not_in_trips_txt)[:5],
        'elapsed_seconds': round(time.time() - t0, 1),
    }
    hourly = {'all': hour_all, 'weekday': hour_wd, 'friday': hour_fr, 'saturday': hour_sa}
    starts = {'all': start_all, 'weekday': start_wd, 'friday': start_fr, 'saturday': start_sa}
    edges = {'all': edges_all, 'weekday': edges_wd, 'friday': edges_fr, 'saturday': edges_sa}
    return edges, hourly, starts, trip_window_mask, stats


print('Streaming ' + STOP_TIMES.name + ' (~15.7M rows, one pass, expect 4-8 minutes) ...')
edge_sets, hourly_counts, start_counts, trip_window_mask, stream_stats = stream_window_edges(
    STOP_TIMES, trip_code)

print()
print('Streaming statistics:')
for k, v in stream_stats.items():
    print('  ' + k + ': ' + str(v))

## 10. נפח שירות לפי שעה ביום השירות

הדבר הראשון שיש להתבונן בו הוא הצורה הגולמית של לוח הזמנים. אנו מטבלים, עבור כל אחת מ-24 השעות של יום השירות, כמה **יציאות מתוכננות מתחנות** מתרחשות (אחת לכל שורה ב-`stop_times.txt` בעלת זמן שמיש) וכמה **נסיעות מתחילות**. שניהם מפולחים לפי סוג-יום. יש לזכור שעמודות סוג-היום חופפות - שירות הפועל מראשון עד שישי תורם גם ל-`weekday` וגם ל-`friday` - ולכן הן אינן חלוקה ממצה ויש לקרוא אותן כ*כמה שירות קיים ביום מייצג מאותו סוג*, ולא כחלק מתוך הסך הכול.

האיור מצלל את חמשת החלונות כדי שניתן יהיה לשפוט את גבולות החלונות מול פרופיל הביקוש בפועל של לוח הזמנים, ומשרטט את נפח יום החול, יום שישי והשבת על אותו ציר. השבת אמורה להיראות קרובה מאוד לרצפה.

In [ ]:
# --- Hourly service volume table ------------------------------------------
hourly_df = pd.DataFrame({
    'hour': list(range(24)),
    'window': [HOUR_TO_WINDOW[h] for h in range(24)],
    'stop_departures_all': hourly_counts['all'],
    'stop_departures_weekday': hourly_counts['weekday'],
    'stop_departures_friday': hourly_counts['friday'],
    'stop_departures_saturday': hourly_counts['saturday'],
    'trips_starting_all': start_counts['all'],
    'trips_starting_weekday': start_counts['weekday'],
    'trips_starting_friday': start_counts['friday'],
    'trips_starting_saturday': start_counts['saturday'],
})
hourly_df.to_csv(TABLES / 'hourly_service_volume.csv', index=False, encoding='utf-8-sig')


def contiguous_runs(hours):
    """Collapse a list of hours into (first, last) runs, for shading a wrapped window."""
    hs = sorted(hours)
    runs, start, prev = [], hs[0], hs[0]
    for h in hs[1:]:
        if h == prev + 1:
            prev = h
        else:
            runs.append((start, prev))
            start = prev = h
    runs.append((start, prev))
    return runs


band_palette = sns.color_palette('pastel', len(WINDOWS))
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

for ax, (col_prefix, ylabel) in zip(
        axes,
        [('stop_departures', 'scheduled stop departures'),
         ('trips_starting', 'trips starting')]):
    for i, (name, s, e) in enumerate(WINDOWS):
        for a, b in contiguous_runs(window_hours(s, e)):
            ax.axvspan(a - 0.5, b + 0.5, color=band_palette[i], alpha=0.45, zorder=0)
    for suffix, style in [('all', '-'), ('weekday', '--'), ('friday', '-.'), ('saturday', ':')]:
        ax.plot(hourly_df['hour'], hourly_df[col_prefix + '_' + suffix],
                style, marker='o', markersize=3, linewidth=1.8, label=suffix, zorder=3)
    ax.set_ylabel(ylabel)
    ax.legend(title='service runs on', ncol=4, loc='upper left')

for i, (name, s, e) in enumerate(WINDOWS):
    for a, b in contiguous_runs(window_hours(s, e)):
        axes[0].text((a + b) / 2.0, axes[0].get_ylim()[1] * 0.02, name,
                     ha='center', va='bottom', fontsize=8, rotation=90, zorder=4)

axes[1].set_xlabel('hour of the service day (GTFS hours >= 24 folded back modulo 24)')
axes[1].set_xticks(range(24))
axes[0].set_title('Scheduled service volume by hour of day, with the five time windows shaded\n'
                  '(SCHEDULED trips - not observed ridership)')
plt.tight_layout()
plt.savefig(FIGURES / 'service_volume_by_hour.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

hourly_df

## 11. בניית גרף אחד לכל חלון

מונה הקטעים המכוון של כל חלון מומר כעת לגרף **בלתי מכוון**, בדיוק באופן שבו מחברת 02 בונה את הרשת הסטטית: שני כיווני הנסיעה של קטע מתמזגים לקשת אחת שמשקלה הוא סכום שתי ספירות המעבר המכוונות. מאפייני התחנות (`stop_name`, `lat`, `lon`, `region`, `metro`) מצורפים מתוך `nodes.csv` של שלב 02, כך שמחברות במורד הזרם מקבלות גרף המתאר את עצמו.

עבור כל חלון אנו מודדים אז:

* **`trips`** - נסיעות התורמות **לפחות קטע אחד** לחלון. נסיעה החוצה גבול חלון נספרת בכל חלון שהיא נוגעת בו, ולכן עמודה זו מסתכמת במכוון ביותר ממספר הנסיעות הייחודיות.
* **`nodes`** - תחנות בעלות לפחות קטע אחד בחלון. תחנה המשורתת רק על ידי נסיעות מחוץ לחלון פשוט אינה מופיעה בגרף של אותו חלון.
* **`directed_edges`** - קטעים מסודרים ייחודיים, בהתאם למוסכמה של `edges.csv` בשלב 02.
* **`avg_degree`** - דרגה בלתי מכוונת ממוצעת, `2m/n`.
* **`largest_component_share`** - שיעור צמתי החלון הנמצאים ברכיב הקשיר הגדול ביותר שלו. זהו מדד הפרגמנטציה: ערך הנמוך במידה ניכרת מ-1 פירושו שהרשת *כבר* מפוצלת עוד לפני שהוסמלה מתקפה כלשהי.

הקבצים `edges_<window>.csv` וקבצי ה-pickle `graph_<window>.pkl` לכל חלון נכתבים כאן; הם החוזה שמחברת 20 תלויה בו.

In [ ]:
# --- Node attributes from stage 02 ----------------------------------------
def _num(value):
    """Coerce to float; return None for blanks, NaN or non-numeric input."""
    if value is None:
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(f) else f


def _txt(value):
    """Coerce to a plain string; NaN and None become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ''
    return str(value)


NODE_ATTR = {}
for rec in nodes_df.to_dict('records'):
    NODE_ATTR[str(rec.get('stop_id'))] = {
        'stop_name': _txt(rec.get('stop_name')),
        'lat': _num(rec.get('lat')),
        'lon': _num(rec.get('lon')),
        'region': _txt(rec.get('region')),
        'metro': _txt(rec.get('metro')),
    }
_DEFAULT_ATTR = {'stop_name': '', 'lat': None, 'lon': None, 'region': '', 'metro': ''}


def build_undirected(edge_dict):
    """Undirected projection of a directed segment counter; weights of both directions summed."""
    G = nx.Graph()
    for (u, v), w in edge_dict.items():
        if G.has_edge(u, v):
            G[u][v]['weight'] += w
        else:
            G.add_edge(u, v, weight=w)
    for n in G.nodes():
        G.nodes[n].update(NODE_ATTR.get(n, _DEFAULT_ATTR))
    return G


def graph_stats(G):
    """nodes, undirected edges, avg degree, largest-component share."""
    n = G.number_of_nodes()
    m = G.number_of_edges()
    if n == 0:
        return {'nodes': 0, 'undirected_edges': 0, 'avg_degree': 0.0,
                'components': 0, 'largest_component_size': 0,
                'largest_component_share': 0.0}
    comps = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    return {
        'nodes': n,
        'undirected_edges': m,
        'avg_degree': round(2.0 * m / n, 3),
        'components': len(comps),
        'largest_component_size': comps[0],
        'largest_component_share': round(comps[0] / n, 4),
    }


# --- Trips per window, from the 5-bit masks collected during the pass ------
trips_per_window = [0] * len(WINDOW_NAMES)
trips_per_window_by_daytype = {dt: [0] * len(WINDOW_NAMES)
                               for dt in ('weekday', 'friday', 'saturday')}
for trip, mask in trip_window_mask.items():
    code = trip_code.get(trip, 0)
    for w in range(len(WINDOW_NAMES)):
        if mask & (1 << w):
            trips_per_window[w] += 1
            if code & BIT_WEEKDAY:
                trips_per_window_by_daytype['weekday'][w] += 1
            if code & BIT_FRIDAY:
                trips_per_window_by_daytype['friday'][w] += 1
            if code & BIT_SATURDAY:
                trips_per_window_by_daytype['saturday'][w] += 1

# --- Build, save and summarise the all-service window graphs ---------------
window_graphs = {}
summary_rows = []
for w, (name, s, e) in enumerate(WINDOWS):
    ed = edge_sets['all'][w]
    G = build_undirected(ed)
    window_graphs[name] = G
    st = graph_stats(G)

    edges_out = pd.DataFrame(
        [{'from_stop': u, 'to_stop': v, 'trip_frequency': int(c)} for (u, v), c in ed.items()]
    ).sort_values('trip_frequency', ascending=False)
    edges_out.to_csv(TABLES / ('edges_' + name + '.csv'), index=False, encoding='utf-8-sig')

    if WRITE_WINDOW_PICKLES:
        with open(STAGE / ('graph_' + name + '.pkl'), 'wb') as fh:
            pickle.dump(G, fh)

    summary_rows.append({
        'window': name,
        'start_hour': s,
        'end_hour': e,
        'trips': trips_per_window[w],
        'nodes': st['nodes'],
        'directed_edges': len(ed),
        'avg_degree': st['avg_degree'],
        'largest_component_share': st['largest_component_share'],
    })
    print(name.ljust(16) + 'nodes ' + format(st['nodes'], '>6,') +
          ' | directed edges ' + format(len(ed), '>7,') +
          ' | avg degree ' + format(st['avg_degree'], '5.2f') +
          ' | LCC share ' + format(st['largest_component_share'], '5.3f') +
          ' | components ' + format(st['components'], '>5,'))

window_summary = pd.DataFrame(summary_rows)
window_summary.to_csv(TABLES / 'window_summary.csv', index=False, encoding='utf-8-sig')
window_summary

## 12. אותם חלונות, בפילוח לפי סוג-יום

הטבלה שלמעלה ממצעת יום שלישי מלא יחד עם שבת כמעט נטולת שירות. כאן אנו בונים מחדש את אותם חמישה חלונות שלוש פעמים נוספות - בהגבלה לנסיעות שהשירות שלהן פועל בימי חול, ביום שישי ובשבת - ומדווחים את אותן סטטיסטיקות בדיוק. גרפים אלה **אינם** נשמרים כ-pickle (מחברת 20 עובדת מתוך חלונות כלל השירות); מטרת סעיף זה היא אבחונית: להראות איזה חלק מהרשת הסטטית הוא למעשה תוצר של יום חול.

תזכורת: שלושת סוגי היום חופפים מעצם ההגדרה, ושירותים בעלי יום `unknown` (ללא דגלים מוגדרים ב-`calendar.txt`) אינם מופיעים באף אחד מהשלושה, אך כן מופיעים בגרפים של כלל השירות שלמעלה.

In [ ]:
# --- Window statistics per day type ---------------------------------------
daytype_rows = []
for daytype in ('weekday', 'friday', 'saturday'):
    for w, (name, s, e) in enumerate(WINDOWS):
        ed = edge_sets[daytype][w]
        st = graph_stats(build_undirected(ed))
        daytype_rows.append({
            'window': name,
            'day_type': daytype,
            'start_hour': s,
            'end_hour': e,
            'trips': trips_per_window_by_daytype[daytype][w],
            'nodes': st['nodes'],
            'directed_edges': len(ed),
            'avg_degree': st['avg_degree'],
            'components': st['components'],
            'largest_component_share': st['largest_component_share'],
        })

daytype_summary = pd.DataFrame(daytype_rows)
daytype_summary.to_csv(TABLES / 'window_summary_by_daytype.csv', index=False, encoding='utf-8-sig')

pivot_nodes = daytype_summary.pivot(index='window', columns='day_type', values='nodes')
pivot_nodes = pivot_nodes.reindex(WINDOW_NAMES)
print('Stations reachable in each window, by day type:')
print(pivot_nodes.to_string())
print()

saturday_total = daytype_summary.loc[daytype_summary['day_type'] == 'saturday', 'directed_edges'].sum()
weekday_total = daytype_summary.loc[daytype_summary['day_type'] == 'weekday', 'directed_edges'].sum()
if weekday_total:
    print('Saturday segment coverage is ' +
          format(100.0 * saturday_total / weekday_total, '.1f') +
          '% of the weekday figure (summed over windows).')

daytype_summary

## 13. כיצד משתנה גודל הרשת בין החלונות

שלוש תצוגות של אותה תוצאה. הפאנל הראשון מציג כמה תחנות וכמה קטעים ייחודיים קיימים בכל חלון - ה*גודל* של הרשת. השני מציג דרגה ממוצעת ואת שיעור הרכיב הגדול ביותר - ה*צורה* של הרשת: האם רשת שעות השפל היא רק עותק קטן יותר של רשת השיא, או רשת שונה מבנית ומפורקת יותר. האיור השלישי חוזר על ספירת התחנות לפי סוג-יום, ושם אפקט השבת בולט בחדות הרבה ביותר.

In [ ]:
# --- Figure: network size across windows ----------------------------------
order = WINDOW_NAMES
ws = window_summary.set_index('window').reindex(order)
x = np.arange(len(order))

fig, ax1 = plt.subplots(figsize=(11, 6))
ax1.bar(x - 0.2, ws['nodes'], width=0.4, label='stations (nodes)', color='#4C72B0')
ax1.set_ylabel('stations in the window graph', color='#4C72B0')
ax1.tick_params(axis='y', labelcolor='#4C72B0')
ax2 = ax1.twinx()
ax2.bar(x + 0.2, ws['directed_edges'], width=0.4, label='directed segments', color='#DD8452')
ax2.set_ylabel('distinct directed segments', color='#DD8452')
ax2.tick_params(axis='y', labelcolor='#DD8452')
ax2.grid(False)
ax1.set_xticks(x)
ax1.set_xticklabels([n.replace('_', ' ') for n in order])
ax1.set_title('Network size by time window (scheduled service, all day types)')
for i, (n, e) in enumerate(zip(ws['nodes'], ws['directed_edges'])):
    ax1.text(i - 0.2, n, format(int(n), ','), ha='center', va='bottom', fontsize=8)
    ax2.text(i + 0.2, e, format(int(e), ','), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / 'network_size_by_window.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

# --- Figure: connectivity across windows ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(x, ws['avg_degree'], color='#55A868')
axes[0].set_xticks(x)
axes[0].set_xticklabels([n.replace('_', ' ') for n in order], rotation=20, ha='right')
axes[0].set_ylabel('average undirected degree')
axes[0].set_title('Average degree by window')
for i, v in enumerate(ws['avg_degree']):
    axes[0].text(i, v, format(v, '.2f'), ha='center', va='bottom', fontsize=9)

axes[1].bar(x, ws['largest_component_share'], color='#C44E52')
axes[1].set_ylim(0, 1.05)
axes[1].set_xticks(x)
axes[1].set_xticklabels([n.replace('_', ' ') for n in order], rotation=20, ha='right')
axes[1].set_ylabel('share of stations in the largest component')
axes[1].set_title('Fragmentation by window')
for i, v in enumerate(ws['largest_component_share']):
    axes[1].text(i, v, format(v, '.3f'), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'connectivity_by_window.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

# --- Figure: weekday vs Friday vs Saturday --------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
width = 0.26
colors = {'weekday': '#4C72B0', 'friday': '#DD8452', 'saturday': '#C44E52'}
for j, metric in enumerate(['nodes', 'directed_edges']):
    piv = daytype_summary.pivot(index='window', columns='day_type', values=metric).reindex(order)
    for k, dt in enumerate(['weekday', 'friday', 'saturday']):
        axes[j].bar(x + (k - 1) * width, piv[dt], width=width, label=dt, color=colors[dt])
    axes[j].set_xticks(x)
    axes[j].set_xticklabels([n.replace('_', ' ') for n in order], rotation=20, ha='right')
    axes[j].set_ylabel(metric.replace('_', ' '))
    axes[j].set_title(metric.replace('_', ' ') + ' by window and day type')
    axes[j].legend(title='service runs on')
plt.tight_layout()
plt.savefig(FIGURES / 'weekday_vs_weekend_by_window.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 14. האם החלונות מכילים את *אותה* רשת, רק דלילה יותר?

הגודל לבדו אינו עונה על השאלה המעניינת. ייתכנו שני חלונות שבכל אחד מהם 40,000 קטעים והם כמעט אינם חולקים אף אחד מהם (רשת שונה באמת) או חולקים כמעט את כולם (אותה רשת בעצימות שונה). אנו מודדים זאת ישירות באמצעות **מדד Jaccard** של קבוצות הקשתות הבלתי מכוונות עבור כל זוג חלונות, `|A and B| / |A or B|`, בתוספת מדד ה*הכלה* (containment) של החלון הקטן יותר בגדול יותר, המצביע על כמה מרשת הלילה היא פשוט תת-קבוצה של שיא הבוקר.

זהו חישוב זול - פעולות קבוצה על לכל היותר כ-50k זוגות לכל חלון.

In [ ]:
# --- Pairwise edge-set overlap between windows ----------------------------
undirected_sets = {
    name: {tuple(sorted((u, v))) for (u, v) in edge_sets['all'][w].keys()}
    for w, (name, _, _) in enumerate(WINDOWS)
}

overlap_rows = []
for a in WINDOW_NAMES:
    for b in WINDOW_NAMES:
        A, B = undirected_sets[a], undirected_sets[b]
        inter = len(A & B)
        union = len(A | B)
        overlap_rows.append({
            'window_a': a,
            'window_b': b,
            'edges_a': len(A),
            'edges_b': len(B),
            'shared_edges': inter,
            'jaccard': round(inter / union, 4) if union else 0.0,
            'share_of_a_covered_by_b': round(inter / len(A), 4) if A else 0.0,
        })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(TABLES / 'window_edge_overlap.csv', index=False, encoding='utf-8-sig')

jac = overlap_df.pivot(index='window_a', columns='window_b', values='jaccard').reindex(
    index=WINDOW_NAMES, columns=WINDOW_NAMES)
cov = overlap_df.pivot(index='window_a', columns='window_b',
                       values='share_of_a_covered_by_b').reindex(
    index=WINDOW_NAMES, columns=WINDOW_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(jac, annot=True, fmt='.2f', cmap='viridis', vmin=0, vmax=1, ax=axes[0],
            cbar_kws={'label': 'Jaccard index'})
axes[0].set_title('Edge-set similarity between windows (Jaccard)')
sns.heatmap(cov, annot=True, fmt='.2f', cmap='magma', vmin=0, vmax=1, ax=axes[1],
            cbar_kws={'label': 'share of row window covered by column window'})
axes[1].set_title('Containment: how much of window A also exists in window B')
for ax in axes:
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIGURES / 'window_edge_overlap.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

jac

## 15. הקטעים העמוסים ביותר בשעת השיא ובחלון הלילה

בדיקה איכותנית מהירה לכך שהחלונות מתנהגים כמצופה: הקטעים המשורתים בכבדות הרבה ביותר בשיא הבוקר אמורים להיראות כמו עורקים עירוניים צפופים, וחלון הלילה אמור להיות נשלט על ידי רשימה קצרה בהרבה של קישורים בין-עירוניים או קישורים מסוג שדה תעופה. שמות התחנות הם בעברית והם מוצגים מימין לשמאל בזכות ה-patch של bidi שהותקן קודם לכן.

In [ ]:
# --- Top segments per window (sanity check) -------------------------------
def top_segments(window_name, k=TOP_N):
    w = WINDOW_INDEX[window_name]
    rows = []
    for (u, v), c in edge_sets['all'][w].items():
        rows.append({
            'from_stop': u,
            'from_name': NODE_ATTR.get(u, _DEFAULT_ATTR)['stop_name'],
            'to_stop': v,
            'to_name': NODE_ATTR.get(v, _DEFAULT_ATTR)['stop_name'],
            'trip_frequency': c,
        })
    return (pd.DataFrame(rows)
            .sort_values('trip_frequency', ascending=False)
            .head(k)
            .reset_index(drop=True))


print('=== Busiest segments, morning_peak ===')
display(top_segments('morning_peak'))
print('=== Busiest segments, night ===')
display(top_segments('night'))

## 16. כתיבת סיכום השלב

כל מה שמחברת מאוחרת יותר או קורא עשויים לרצות כמספר בודד נאסף לתוך `window_construction_summary.json`: הגדרת החלונות שנעשה בה שימוש בפועל, סטטיסטיקות מעבר ה-streaming (שורות שנקראו, קטעים שהוקצו, שורות בעלות שעות שאחרי חצות, זמן ריצה), הסטטיסטיקות לכל חלון, מפקד הנסיעות לפי סוג-יום, וההסתייגויות המפורשות. הרישום הסופי מאשר אילו קבצים נכתבו ומה גודלם.

In [ ]:
# --- Stage summary --------------------------------------------------------
summary = {
    'stage': '19_time_of_day_graphs',
    'source': 'GTFS stop_times.txt + trips.txt + calendar.txt (SCHEDULED service, not ridership)',
    'windows': [{'window': n, 'start_hour': s, 'end_hour': e,
                 'hours': window_hours(s, e)} for n, s, e in WINDOWS],
    'segment_assignment_rule': 'departure time at the ORIGIN stop of the segment, hour taken modulo 24',
    'streaming_pass': stream_stats,
    'calendar': {
        'services_total': int(n_services),
        'services_with_no_day_flag': int(n_unknown_services),
        'trips_total': int(n_trips_total),
        'trips_by_day_type_overlapping': {k: int(v) for k, v in trip_day_counts.items()},
        'calendar_dates_txt_present': (DATA / 'calendar_dates.txt').exists(),
    },
    'per_window': {
        r['window']: {k: (float(r[k]) if k in ('avg_degree', 'largest_component_share')
                          else int(r[k]))
                      for k in ('trips', 'nodes', 'directed_edges',
                                'avg_degree', 'largest_component_share')}
        for r in summary_rows
    },
    'caveats': [
        'All counts are SCHEDULED trips from the GTFS timetable. They are not observed '
        'ridership, boardings, or vehicle occupancy; nothing in this feed measures demand.',
        'A trip that crosses a window boundary is counted in every window it touches, so the '
        'trips column sums to more than the number of distinct trips.',
        'Day types (weekday / friday / saturday) are not mutually exclusive: a service running '
        'Sunday-Friday is counted under both weekday and friday.',
        'Services with all seven day flags set to 0 have an unknown day of week because this '
        'feed ships no calendar_dates.txt; they are excluded from the day-type tables but '
        'included in the all-service window graphs.',
        'GTFS hours >= 24 are folded modulo 24, so a 25:30 departure is treated as 01:30 for '
        'the purpose of window assignment.',
    ],
}

with open(STAGE / 'window_construction_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('Files written under ' + str(STAGE) + ':')
for p in sorted(STAGE.rglob('*')):
    if p.is_file():
        print('  ' + str(p.relative_to(STAGE)).ljust(46) +
              format(p.stat().st_size / 1024, '>10,.0f') + ' KB')

## מסקנות

יש לקרוא אותן אל מול המספרים שהודפסו בפועל למעלה - הניסוח להלן מתאר את מה שהניתוח יכול ואינו יכול לתמוך בו, ולא מסקנה שנקבעה מראש.

1. **הרשת הסטטית של מחברות 02-13 היא איחוד, לא תצלום רגע.** אף רגע בודד בשבוע אינו נראה כמו הגרף שיתר הפרויקט מנתח. איחוד כל החלונות מכיל כל קטע הפועל בשעה כלשהי ביום כלשהו; כל גרף חלון בנפרד קטן ממנו ממש, הן במספר תחנות והן במספר קטעים. לפיכך, כל טענת חוסן הנטענת על הגרף הסטטי היא טענה לגבי *הגרסה הקשירה ביותר* של הרשת, שהיא המקרה האופטימי.

2. **ההידרדרות בשעות השפל היא בעיקר דילול, ובחלקה פרגמנטציה של ממש.** יש להשוות בין `avg_degree` ובין `largest_component_share` על פני החלונות ב-`window_summary.csv`. במקום שבו שיעור הרכיב הגדול ביותר יורד, הרשת אינה רק מפעילה פחות אוטובוסים - היא התפרקה בפועל לחלקים שאינם יכולים להגיע זה לזה באותה שעה, וזו טענה חזקה ממש מאשר תדירות נמוכה.

3. **חלון הלילה הוא רשת שונה, ולא רשת קטנה יותר.** מפת החום של Jaccard מכמתת זאת: חלונות יום סמוכים חופפים במידה רבה (הם אותם עורקים בעצימויות שונות), בעוד ש-`night` חולק חלק קטן יחסית מקבוצת הקשתות שלו עם שעות השיא ובעיקר *מוכל* בהן ולא דומה להן.

4. **השבת היא האפקט המבני הגדול ביותר בכל הפיד.** עמודות יום השבת ב-`window_summary_by_daytype.csv` הן שבריר קטן מעמודות ימי החול. מיצוע השבוע לגרף אחד, כפי שעושה כל מחברת קודמת, מערבב אפוא רשת מלאת שירות עם רשת כמעט נעדרת. כל מסקנה בדבר שוויוניות או נגישות הנגזרת מהגרף הסטטי מניחה במובלע שירות של יום חול.

5. **מה שמחברת זו אינה מראה.** מדובר בנסיעות **מתוכננות**. קטע בעל `trip_frequency` גבוה הוא קטע שלוח הזמנים משרת לעיתים קרובות, וזהו פרוקסי ל- ולא מדידה של - חשיבות עבור הנוסעים. אין בפיד זה נתוני נסועה, תפוסה או אמינות, ולכן קטע בעל תדירות נמוכה המשרת אוכלוסייה נטולת חלופות וקטע בעל תדירות גבוהה שיש לו חלופות אינם ניתנים להבחנה כאן. מחברת 21 מתמודדת עם פער זה באמצעות פרוקסי ביקוש מבוסס אוכלוסייה, וגם הוא פרוקסי.

6. **שתי מגבלות כנות של הבנייה עצמה.** (א) השתייכות לחלון נקבעת לפי זמן היציאה בתחנת *המוצא* של הקטע, ולכן נסיעה בין-עירונית ארוכה המתחילה ב-08:50 מיוחסת כולה לשיא הבוקר אף שרובה מתרחש אחרי 09:00; ברוחבי חלון של 3-6 שעות הדבר משפיע רק על מיעוט קטן של קטעים, אך זהו קירוב ממשי. (ב) מכיוון שלפיד זה אין `calendar_dates.txt`, לשירותים שכל דגלי השבוע שלהם אפס לא ניתן לשייך יום בשבוע כלל; הם מדווחים כ-`unknown` ולא מקופלים בשקט לתוך נתוני ימי החול.

**מסירה למחברת 20:** `graph_<window>.pkl` ו-`tables/edges_<window>.csv` הם הרשתות לכל חלון; `tables/window_summary.csv` היא טבלת החוזה. מחברת 20 מחשבת מחדש centrality ומדמה הסרה ממוקדת בתוך כל חלון, כדי לבחון האם קבוצת התחנות הקריטיות יציבה לאורך היום או שמא הקריטיות עצמה תלוית זמן.